In [ ]:
# Get all json files from in/
import os
import datetime
import json
import plotly.graph_objs as go
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Helles Plotly-Theme erzwingen (unabhängig vom IDE-/Notebook-Theme)
pio.templates.default = "plotly_white"


In [ ]:
out_dir = "out/"
in_dir = "inStewa/"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
files = [f for f in os.listdir(in_dir) if f.endswith('.csv')]

len(files)

In [ ]:
ret_df = []

for f in files:
    df = pd.read_csv(in_dir + f, sep=';')
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    ret_df.append(df)


In [ ]:
# merge
df = pd.concat(ret_df)
# sort
df = df.sort_values(by='Timestamp')

df.columns

In [ ]:
# start_date = "2025-02-19"
start_date = "2026-07-10"
end_date = "2026-08-11"

start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date) + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)

plt_global_title = f"[{start_date.strftime('%d.%m.%Y')} - {end_date.strftime('%d.%m.%Y')}]"

df = df[(df["Timestamp"] >= start_date) & (df["Timestamp"] <= end_date)].copy()


In [ ]:
plt_global_title

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=df['Timestamp'], y=df['Anzahl'], name='alle', marker_color='blue', opacity=1.0))
fig.add_trace(go.Bar(x=df['Timestamp'], y=df['Kritisch'], name='kritisch', marker_color='red', opacity=1.0))

fig.update_layout(
    title='Detektionen',
    xaxis_title='Zeit (UTC)',
    yaxis_title='Anzahl (in einer Stunde)',
    barmode='overlay'  # Überlappen statt Stapeln
)

fig.show()

fig.write_html(f"{out_dir}/report-detektionen.html")


In [ ]:
# Count by hour of the day
def count_by(attribute="hour"):
    df['hour'] = df['Timestamp'].dt.hour
    df['day'] = df['Timestamp'].dt.day
    # Day of the week
    df['dayofweek'] = df['Timestamp'].dt.dayofweek
    # Month
    df['month'] = df['Timestamp'].dt.month

    df_hour_anzahl = df.groupby(attribute)['Anzahl'].sum().reset_index()
    df_hour_kritisch = df.groupby(attribute)['Kritisch'].sum().reset_index()
    # Sort by hour
    df_hour_anzahl = df_hour_anzahl.sort_values(by=attribute)
    df_hour_kritisch = df_hour_kritisch.sort_values(by=attribute)

    fig = go.Figure()
    fig.add_trace(
        go.Bar(x=df_hour_anzahl[attribute], y=df_hour_anzahl['Anzahl'], name='alle', marker_color='blue', opacity=1.0))
    fig.add_trace(
        go.Bar(x=df_hour_kritisch[attribute], y=df_hour_kritisch['Kritisch'], name='kritisch', marker_color='red',
               opacity=1.0))
    fig.update_layout(
        title='Meteor Detektionen nach ' + attribute,
        xaxis_title='' + attribute,
        yaxis_title='Anzahl',
        barmode='overlay'
    )

    if attribute == "dayofweek":
        fig.update_xaxes(
            tickvals=[0, 1, 2, 3, 4, 5, 6],
            ticktext=['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']
        )

    if attribute == "month":
        fig.update_xaxes(
            tickvals=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            ticktext=['Jan', 'Feb', 'Mär', 'Apr', 'Mai', 'Jun', 'Jul', 'Aug', 'Sep', 'Okt', 'Nov', 'Dez']
        )

    if attribute == "hour":
        fig.update_layout(xaxis_title='Stunde [UTC]',
                          title=f'Detektionen<br>{plt_global_title}')

    # Fixed plt size
    fig.update_layout(
        width=800,
        height=400
    )
    fig.show()

    fig.write_html(f"{out_dir}/report-count-{attribute}.html")


count_by("hour")
#count_by("day")
count_by("dayofweek")
count_by("month")

In [ ]:
def heatmap_day_hour(df,
                     ):
    df['hour'] = df['Timestamp'].dt.hour
    df['dayofweek'] = df['Timestamp'].dt.dayofweek

    # Gruppieren nach Wochentag und Stunde und Summieren der Werte
    heatmap_data = df.groupby(['dayofweek', 'hour'])['Anzahl'].sum().reset_index()

    # Pivot-Tabelle für die Heatmap
    heatmap_pivot = heatmap_data.pivot(index='dayofweek', columns='hour', values='Anzahl')

    fig = px.imshow(
        heatmap_pivot,
        labels={'x': 'Stunde', 'y': 'Wochentag', 'color': 'Anzahl'},
        x=heatmap_pivot.columns,
        y=['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag'],
        # color_continuous_scale='Viridis'
    )

    fig.update_layout(
        title=f"Detektionen<br>{plt_global_title}",
        xaxis_title='Stunde [UTC]',
        yaxis_title='Wochentag',
        width=800,
        height=500
    )

    fig.show()

    fig.write_html(f"{out_dir}/report-heatmap-day-hour.html")


# Beispielaufruf
heatmap_day_hour(df)

In [ ]:
def plot_hour_day_heatmap(df,
                          # start_date="1999-06-06",
                          # end_date="2035-06-23",
                          id_field="Anzahl",
                          vmin=None,
                          vmax=None,
                          out_html_fp=None,
                          out_pdf_fp=None,
                          plt_title_override=""
                          ):
    df = df.copy()

    # Sicherstellen, dass 'Timestamp' ein datetime-Objekt ist
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df = df[df['Timestamp'].notna()]

    # Neue Spalten fuer Stunde und Datum
    df['hour'] = df['Timestamp'].dt.hour.astype(int)
    df['date'] = df['Timestamp'].dt.date

    # Anzahl pro Datum und Stunde summieren
    heatmap_data = df.groupby(['date', 'hour'])[id_field].sum().unstack(fill_value=0)
    heatmap_data = heatmap_data.sort_index()

    # Alle Stunden anzeigen, auch wenn eine Stunde keine Eintraege hat
    heatmap_data = heatmap_data.reindex(columns=range(24), fill_value=0)

    # start_date = pd.to_datetime(start_date).date()
    # end_date = pd.to_datetime(end_date).date()
    # heatmap_data = heatmap_data.loc[
    #     (heatmap_data.index >= start_date) & (heatmap_data.index <= end_date)
    #     ]

    heatmap_data.index = pd.to_datetime(heatmap_data.index).strftime('%d.%m.%Y')

    fig = px.imshow(
        heatmap_data,
        labels={'x': 'Hour', 'y': 'Date', 'color': id_field},
        x=heatmap_data.columns,
        y=heatmap_data.index,
        color_continuous_scale='Viridis',
        text_auto=True,
        title=f'Heatmap: [{id_field}]',
        zmin=vmin,
        zmax=vmax,
    )
    fig.update_layout(
        xaxis_title='Stunde [UTC]',
        yaxis_title='Datum',
        width=1200,
        height=max(500, 28 * len(heatmap_data)),
    )
    if plt_title_override != "":
        fig.update_layout(
            title=plt_title_override,
        )

    fig.show()

    if out_html_fp:
        fig.write_html(out_html_fp)

    if out_pdf_fp:
        fig.write_image(out_pdf_fp)






In [ ]:
plot_hour_day_heatmap(df,
                      # start_date="2025-02-19",
                      # end_date="2026-07-21",
                      vmin=0,
                      # vmax=400,
                      vmax=250,
                      id_field="Anzahl",
                      out_html_fp=f"{out_dir}report-heatmap-anzahl.html",
                      plt_title_override=f"Detektionen<br>{plt_global_title}",
                      )

In [ ]:
plot_hour_day_heatmap(df,
                      # start_date="2025-02-19",
                      # end_date="2026-07-21",
                      vmin=0,
                      vmax=300,
                      id_field="Kritisch",
                      out_html_fp=f"{out_dir}report-heatmap-kritisch.html",
                      )
